# Random Forest de Momentum — Direção do AUDUSD

Classificação **binária** da direção do próximo candle do **AUDUSD** a partir de *features* de momentum. O modelo é um `RandomForestClassifier` e a posição é dimensionada pelo inverso da volatilidade ($w_t = 1/\sigma_t$).

> Este `.ipynb` é um **exemplo/fixture** do template. Converta-o com `quant/templates/convert.sh` para ver o visual.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Base privada: a string de conexão vem do ambiente, nunca do notebook.
prices = load_prices(os.environ["QUANT_DB_URL"], symbol="AUDUSD", timeframe="15min")
print(f"{len(prices):,} candles · {prices.index[0]:%Y-%m} → {prices.index[-1]:%Y-%m} · fonte: base privada (oculta)")

18,432 candles · 2021-01 → 2024-12 · fonte: base privada (oculta)


## Feature engineering

Retorno, momentum de 5 e 20 períodos e volatilidade móvel (20). O alvo é a direção do próximo candle.

In [2]:
def make_features(df):
    df["ret"]   = df["close"].pct_change()
    df["mom5"]  = df["close"].pct_change(5)
    df["mom20"] = df["close"].pct_change(20)
    df["vol"]   = df["ret"].rolling(20).std()
    df["target"] = (df["close"].shift(-1) > df["close"]).astype(int)
    return df.dropna()

data = make_features(prices)
data[["ret", "mom5", "mom20", "vol", "target"]].head(3)

,ret,mom5,mom20,vol,target
20,0.00012,0.00231,0.00518,0.00104,1
21,-0.00008,0.00187,0.00472,0.00101,0
22,0.00021,0.00203,0.00489,0.00099,1


## Treino do modelo

Split temporal 70/30 (sem embaralhar, para não vazar futuro). Random forest com profundidade limitada.

In [3]:
features = ["mom5", "mom20", "vol"]
split = int(len(data) * 0.7)
X_tr, y_tr = data[features][:split], data["target"][:split]
X_te, y_te = data[features][split:], data["target"][split:]

model = RandomForestClassifier(n_estimators=300, max_depth=5, random_state=42)
model.fit(X_tr, y_tr)
acc = accuracy_score(y_te, model.predict(X_te))
print(f"Acurácia out-of-sample: {acc:.3f}")

Acurácia out-of-sample: 0.571


## Backtest e gestão de risco

Posição *long* quando o sinal é 1, dimensionada pelo inverso da volatilidade. Métricas sobre o período de teste (out-of-sample).

In [4]:
from IPython.display import HTML

signal = model.predict(X_te)
ret    = data["ret"][split:].values
size   = 1 / data["vol"][split:].values          # risco-alvo por volatilidade
pnl    = signal * ret * np.clip(size, 0, 3)
equity = (1 + pnl).cumprod()

HTML(render_equity_card(equity, acc))            # helper interno -> SVG read-only

## Conclusão

Edge modesto, porém consistente *out-of-sample*, com drawdown controlado pelo *sizing* por volatilidade. Próximos passos:

- validação *walk-forward* (sem vazamento de futuro);
- custos de transação e *slippage*;
- *ensembling* com o notebook de RSI.